In [ ]:
import pandas as pd
import glob
import os

files = sorted(glob.glob('D:/Lomba/DATATHON DICODING/inflasiwatch/data/raw/IHK_Kel_06/*.csv'))
for f in files:
    try:
        df = pd.read_csv(f, header=None)
        header_row = -1
        for i, r in df.head(5).iterrows():
            if 'Januari' in list(r.values):
                header_row = i
                break
        print(f"File: {f.split('/')[-1]}, Header row: {header_row}")
    except Exception as e:
        print(f"Error {f}: {e}")

# Kota target
target_cities = [
    r"KOTA SURABAYA",
    r"(KOTA|DKI)\s+JAKARTA",
    r"KOTA MAKASSAR",
    r"KOTA MEDAN",
    r"KOTA BANDUNG"
]

# Mapping bulan Indonesia
month_map = {
    "Januari": 1,
    "Februari": 2,
    "Maret": 3,
    "April": 4,
    "Mei": 5,
    "Juni": 6,
    "Juli": 7,
    "Agustus": 8,
    "September": 9,
    "Oktober": 10,
    "November": 11,
    "Desember": 12
}

all_data = []

File: IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2020.csv, Header row: 4
File: IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2021.csv, Header row: 4
File: IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2022.csv, Header row: 4
File: IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2023.csv, Header row: 4
File: IHK_Kel_06\Indeks Harga Konsumen (2022=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2024.csv, Header row: 4
File: IHK_Kel_06\Indeks Harga Konsumen (2022=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2025.csv, Header row: 4
File: IHK_Kel_06\Indeks Harga Konsumen (2022=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2026.csv, Header row: 4


In [2]:
for file in files:
    print(f"Reading {file}")
    
    df = pd.read_csv(file, header=4)

    
    # rename kolom pertama jadi kota kalau belum ada nama
    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Kota"})
    
    # filter kota target
    city_rows = df[
        df["Kota"].astype(str).str.upper().str.contains(
            "|".join(target_cities),
            na=False
        )
    ].copy()
    
    # ambil tahun dari nama file
    year = int(os.path.basename(file).split(",")[-1].replace(".csv", ""))
    
    # melt bulan
    months = [m for m in month_map.keys() if m in city_rows.columns]
    
    city_long = city_rows.melt(
        id_vars=["Kota"],
        value_vars=months,
        var_name="month",
        value_name="ihk"
    )
    
    city_long["date"] = pd.to_datetime({
        "year": year,
        "month": city_long["month"].map(month_map),
        "day": 1
    })
    
    all_data.append(city_long[["Kota", "date", "ihk"]])

Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2020.csv
Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2021.csv
Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2022.csv
Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2018=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2023.csv
Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2022=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2024.csv
Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2022=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2025.csv
Reading D:/Lomba/DATATHON DICODING/IHK_Kel_06\Indeks Harga Konsumen (2022=100) Menurut Kelompok dan Sub Kelompok 06 Transportasi, 2026.csv


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8772\3050666242.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Kota"].astype(str).str.upper().str.contains(
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8772\3050666242.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Kota"].astype(str).str.upper().str.contains(
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8772\3050666242.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Kota"].astype(str).str.upper().str.contains(
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_8772\3050666242.py:13: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df["Kota"].astype(str).str.upper().str.co

In [3]:
final_df = pd.concat(all_data, ignore_index=True)

final_df["Kota"] = final_df["Kota"].replace("DKI JAKARTA", "KOTA JAKARTA")

final_df = final_df.sort_values(["Kota", "date"])

print(final_df.head())
print(final_df.tail())

            Kota       date     ihk
2   KOTA BANDUNG 2020-01-01  101.12
7   KOTA BANDUNG 2020-02-01  100.39
12  KOTA BANDUNG 2020-03-01   99.79
17  KOTA BANDUNG 2020-04-01   99.41
22  KOTA BANDUNG 2020-05-01   99.53
              Kota       date ihk
398  KOTA SURABAYA 2026-08-01   -
403  KOTA SURABAYA 2026-09-01   -
408  KOTA SURABAYA 2026-10-01   -
413  KOTA SURABAYA 2026-11-01   -
418  KOTA SURABAYA 2026-12-01   -


In [ ]:
output_file = r"D:/Lomba/DATATHON DICODING/inflasiwatch/data/processed/ihk/IHK_Kel_06_5_Kota.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for city in final_df["Kota"].unique():
        city_df = final_df[final_df["Kota"] == city]
        
        sheet_name = city[:31]
        city_df.to_excel(writer, sheet_name=sheet_name, index=False)

print("Done saving")

Done saving
